In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PySparkTransformations_Actions").getOrCreate()

In [32]:
from google.colab import files

uploaded = files.upload()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000.csv
Saving lazy_execution_products.csv to lazy_execution_products.csv


In [33]:
df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)

**Part A - Basic Transformations and Actions**

In [34]:
# 1. Filter all transactions where quantity > 5 . Store the result in a new DataFrame without displaying it.
high_qty_df = df.filter(df.quantity >= 5)

In [35]:
# 2. Select only order_id , product_id , quantity , and unit_price . Use show() to trigger execution.
high_qty_df.select("order_id", "product_id", "quantity", "unit_price").show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000009|      P007|       5|     79999|
| O000010|      P006|       5|     55999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000019|      P014|       5|     29999|
| O000020|      P010|       7|      3999|
| O000021|      P006|       7|     55999|
| O000022|      P003|       7|      1899|
| O000023|      P010|       9|      3999|
| O000024|      P009|       7|      1999|
+--------+----------+--------+----

In [36]:
# 3. Filter transactions with unit_price > 20000 . Use count() to determine the number of matching records.
expensive_items_count = df.filter(df.unit_price > 20000).count()
print(f"Number of matching records: {expensive_items_count}")

Number of matching records: 6560


In [37]:
# 4. Create a new column gross_amount by multiplying quantity * unit_price. Inspect the execution plan using explain() before displaying the results.
gross_df = df.withColumn("gross_amount", df.quantity * df.unit_price)

gross_df.explain()
gross_df.show()

== Physical Plan ==
*(1) Project [order_id#696, order_date#697, store_id#698, product_id#699, quantity#700, unit_price#701, discount_pct#702, payment_method#703, order_status#704, sales_channel#705, (quantity#700 * unit_price#701) AS gross_amount#739]
+- FileScan csv [order_id#696,order_date#697,store_id#698,product_id#699,quantity#700,unit_price#701,discount_pct#702,payment_method#703,order_status#704,sales_channel#705] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------

In [44]:
# 5. Filter transactions with order_status = "Completed" and sort them by unit_price descending. Trigger execution using take(10).
completed_top10 = df.filter(df.order_status == "Completed").orderBy(df.unit_price.desc()).take(10)
print(completed_top10)

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'), Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'), Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, un

**Part B — Chained transformations**

In [45]:
from pyspark.sql.functions import sum, col

In [48]:
# 6. Create a pipeline that filters completed orders, calculates gross amount, selects five relevant columns, and sorts by gross amount descending. Use only one action at the end.
pipeline_df = df.filter(df.order_status == "Completed") \
                .withColumn("gross_amount", df.quantity * df.unit_price) \
                .select("order_id", "store_id", "quantity", "unit_price", "gross_amount") \
                .orderBy(col("gross_amount").desc())

pipeline_df.show()

+--------+--------+--------+----------+------------+
|order_id|store_id|quantity|unit_price|gross_amount|
+--------+--------+--------+----------+------------+
| O002603|     S01|       9|     79999|      719991|
| O006170|     S04|       9|     79999|      719991|
| O002693|     S07|       9|     79999|      719991|
| O000107|     S05|       9|     79999|      719991|
| O002841|     S02|       9|     79999|      719991|
| O000367|     S01|       9|     79999|      719991|
| O002940|     S02|       9|     79999|      719991|
| O000527|     S02|       9|     79999|      719991|
| O003108|     S08|       9|     79999|      719991|
| O000690|     S02|       9|     79999|      719991|
| O003542|     S04|       9|     79999|      719991|
| O001201|     S05|       9|     79999|      719991|
| O003884|     S06|       9|     79999|      719991|
| O001300|     S03|       9|     79999|      719991|
| O003897|     S02|       9|     79999|      719991|
| O001796|     S03|       9|     79999|      7

In [49]:
# 7. Filter orders with discounts of at least 10%, calculate the discounted sales amount, and display the top 15 orders.
discounted_df = df.filter(df.discount_pct >= 10) \
                  .withColumn("discounted_amount", (df.quantity * df.unit_price) * (1 - (df.discount_pct / 100))) \
                  .orderBy(col("discounted_amount").desc())

discounted_df.show()

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|         647991.9|
| O018838|2026-06-21|     S01|      P007|       9|     79999|          10|          Cash|   Completed|       Online|         647991.9|
| O007743|2026-05-22|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|         647991.9|
| O000264|2026-02-24|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|         647991.9|
| O009625|2026-05-02|     S03|      P007|       9|     

In [50]:
# 8. Group transactions by payment_method , calculate total quantity sold, and sort the results in descending order.
payment_summary_df = df.groupBy("payment_method") \
                       .agg(sum("quantity").alias("total_quantity_sold")) \
                       .orderBy(col("total_quantity_sold").desc())

payment_summary_df.show()

+--------------+-------------------+
|payment_method|total_quantity_sold|
+--------------+-------------------+
|          Cash|              31754|
|          Card|              31327|
|           UPI|              30894|
|   Net Banking|              30889|
+--------------+-------------------+



In [54]:
# 9. Group transactions by sales_channel and order_status , then calculate the transaction count for each combination.
channel_status_df = df.groupBy("sales_channel", "order_status").count()

channel_status_df.show()

+-------------+------------+-----+
|sales_channel|order_status|count|
+-------------+------------+-----+
|       Online|   Completed|10797|
|     In Store|   Cancelled|  592|
|       Online|    Returned|  987|
|       Online|   Cancelled|  633|
|     In Store|    Returned| 1013|
|     In Store|   Completed|10978|
+-------------+------------+-----+



In [52]:
# 10. Create a DataFrame containing only order_id , store_id , product_id , and quantity. Use drop() to remove store_id , then use first() to retrieve one record.
subset_df = df.select("order_id", "store_id", "product_id", "quantity")

first_record = subset_df.drop("store_id").first()
print(first_record)

Row(order_id='O000001', product_id='P002', quantity=3)


**Part C — Joins and aggregations**

In [60]:
sales_df = spark.read.csv("lazy_execution_sales_25000.csv", header=True, inferSchema=True)
products_df = spark.read.csv("lazy_execution_products.csv", header=True, inferSchema=True)
stores_df = spark.read.csv("lazy_execution_stores.csv", header=True, inferSchema=True)

In [61]:
# 11. Join sales_df with products_df using product_id . Select product name, category, quantity, and unit price. Display 20 records.
joined_products_df = sales_df.join(products_df, on="product_id", how="inner").select("product_name", "category", "quantity", "unit_price")

joined_products_df.show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [62]:
# 12. Join sales_df with stores_df using store_id . Group by city and calculate the total quantity sold.
joined_stores_df = sales_df.join(stores_df, on="store_id", how="inner").groupBy("city").agg(sum("quantity").alias("total_quantity_sold"))

joined_stores_df.show()

+---------+-------------------+
|     city|total_quantity_sold|
+---------+-------------------+
|  Chennai|              15727|
|   Mumbai|              15489|
|  Kolkata|              15619|
|     Pune|              15407|
|    Delhi|              15101|
|Bengaluru|              15718|
|Hyderabad|              15784|
|   Jaipur|              16019|
+---------+-------------------+



In [64]:
# 13. Join all three DataFrames. Calculate total completed-order revenue for each product category and region.
three_way_join_df = sales_df.join(products_df, on="product_id").join(stores_df, on="store_id")
revenue_df = three_way_join_df.filter(col("order_status") == "Completed").withColumn("revenue", col("quantity") * col("unit_price")).groupBy("category", "region").agg(sum("revenue").alias("total_revenue"))

revenue_df.show()


+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [66]:
# 14. Filter completed transactions, join the product master, calculate discounted revenue, and display the top five products by total revenue.
top_products_df = sales_df.filter(col("order_status") == "Completed") \
                          .join(products_df, on="product_id") \
                          .withColumn("discounted_revenue", (col("quantity") * col("unit_price")) * (1 - (col("discount_pct") / 100))) \
                          .groupBy("product_name") \
                          .agg(sum("discounted_revenue").alias("total_revenue")) \
                          .orderBy(col("total_revenue").desc())

top_products_df.show(5)

+---------------+--------------------+
|   product_name|       total_revenue|
+---------------+--------------------+
|      Laptop 16|5.2272546585000014E8|
|      Laptop 14| 3.830051604999999E8|
|  Smartphone 5G| 2.060406317500002E8|
| Android Tablet|1.5040656029999995E8|
|27-inch Monitor|1.1406074015000013E8|
+---------------+--------------------+
only showing top 5 rows


**Part D — Understanding lazy execution**

In [81]:
# 15. Create five consecutive transformations using filter(), select(), withColumn(), groupBy(), and orderBy(). Add a Python print() after each transformation. Observe when Spark actually processes the data.
print("Starting pipeline setup...")

step1 = df.filter(df.order_status == "Completed")
print("Step 1: Filter applied.")

step2 = step1.select("order_id", "store_id", "quantity", "unit_price")
print("Step 2: Columns selected.")

step3 = step2.withColumn("revenue", col("quantity") * col("unit_price"))
print("Step 3: Revenue column added.")

step4 = step3.groupBy("store_id")
print("Step 4: Grouped by Store ID.")

step5 = step4.agg(sum("unit_price").alias("total_price")).orderBy("total_price")
print("Step 5: Aggregated and sorted.")

step1.show()
step2.show()
step3.show()
step5.show()
print("Pipeline execution complete.")

Starting pipeline setup...
Step 1: Filter applied.
Step 2: Columns selected.
Step 3: Revenue column added.
Step 4: Grouped by Store ID.
Step 5: Aggregated and sorted.
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026

In [76]:
# 16. Create a filtered DataFrame and inspect it with explain() . Then call show() and count() separately. Observe the Spark jobs in the Colab output.
filtered_df = df.filter(df.order_status == "Completed")

filtered_df.explain()

filtered_df.show()

filtered_df.count()

== Physical Plan ==
*(1) Filter (isnotnull(order_status#704) AND (order_status#704 = Completed))
+- FileScan csv [order_id#696,order_date#697,store_id#698,product_id#699,quantity#700,unit_price#701,discount_pct#702,payment_method#703,order_status#704,sales_channel#705] Batched: false, DataFilters: [isnotnull(order_status#704), (order_status#704 = Completed)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [IsNotNull(order_status), EqualTo(order_status,Completed)], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------

21775

In [78]:
# 18. Filter completed orders, calculate discounted revenue, and save the resulting DataFrame as Parquet using write.mode("overwrite").parquet().
output_df = df.filter(df.order_status == "Completed").withColumn("discounted_revenue", (df.quantity * df.unit_price) * (1 - (df.discount_pct / 100)))

output_df.write.mode("overwrite").parquet("output/completed_orders.parquet")

17. Use collect() to retrieve the grouped total quantity for each product category. Explain why
collecting a small aggregated result is safer than collecting all 25,000 rows.

- Using `collect()` on grouped data by product category is safe because it returns only a few rows, which can easily fit in the driver's memory. However, using `collect()` on all 25000 raw rows can use a lot of memory because all the data is transferred to the driver. If the dataset grows to millions of rows, it may cause an OutOfMemory (OOM) error and crash the Spark application.